# Aula 08: MongoDB Aggregation Framework
### Tecnologia em Análise e Desenvolvimento de Sistemas (ADS) — IFCE Campus Tauá
**Componente:** Banco de Dados Não-Relacionais (ADS16)  
**Docente:** Prof. Me. Reginaldo Fernandes

---

## 🎯 Objetivos Deste Caderno Interativo
1. Compreender a arquitetura de **Pipeline de Agregação (Dataflow)** do MongoDB.
2. Comparar operadores de agregação do MongoDB com cláusulas SQL (`WHERE`, `GROUP BY`, `HAVING`, `JOIN`).
3. Executar pipelines analíticos reais sobre o catálogo e vendas do e-commerce *ShopGrids*.
4. Medir o impacto de otimizações de memória RAM e filtros precoces com `$match`.

In [ ]:
from pymongo import MongoClient
import pandas as pd

# Conectando à instância local do MongoDB
client = MongoClient("mongodb://localhost:27017", serverSelectionTimeoutMS=2000)
db = client["ecommerce_shopgrids"]
print("Coleções disponíveis:", db.list_collection_names())

## 1. Faturamento Mensal e Ticket Médio
Uso encadeado dos estágios `$match` $\to$ `$group` $\to$ `$project` $\to$ `$sort`.

In [ ]:
pipeline_faturamento = [
    {"$match": {"status": "entregue"}},
    {
        "$group": {
            "_id": {"ano": {"$year": "$data_pedido"}, "mes": {"$month": "$data_pedido"}},
            "faturamento": {"$sum": "$valor_total"},
            "lucro": {"$sum": "$lucro_bruto"},
            "ticket_medio": {"$avg": "$valor_total"},
            "total_pedidos": {"$sum": 1}
        }
    },
    {
        "$project": {
            "_id": 0,
            "ano": "$_id.ano",
            "mes": "$_id.mes",
            "faturamento": {"$round": ["$faturamento", 2]},
            "ticket_medio": {"$round": ["$ticket_medio", 2]},
            "total_pedidos": 1
        }
    },
    {"$sort": {"ano": 1, "mes": 1}}
]

df_faturamento = pd.DataFrame(list(db.pedidos.aggregate(pipeline_faturamento)))
df_faturamento

## 2. Desconstrução de Arrays com `$unwind`: Produtos Mais Vendidos
Cada documento de pedido contém uma lista de itens. O `$unwind` expande a lista em fluxos individuais.

In [ ]:
pipeline_produtos = [
    {"$match": {"status": "entregue"}},
    {"$unwind": "$itens"},
    {
        "$group": {
            "_id": {"sku": "$itens.sku", "produto": "$itens.nome", "categoria": "$itens.categoria"},
            "quantidade": {"$sum": "$itens.quantidade"},
            "receita": {"$sum": "$itens.subtotal"}
        }
    },
    {
        "$project": {
            "_id": 0,
            "sku": "$_id.sku",
            "produto": "$_id.produto",
            "categoria": "$_id.categoria",
            "quantidade": 1,
            "receita": {"$round": ["$receita", 2]}
        }
    },
    {"$sort": {"receita": -1}},
    {"$limit": 5}
]

pd.DataFrame(list(db.pedidos.aggregate(pipeline_produtos)))

## 3. Junção Externa com `$lookup` e Avaliação de Catálogo
Cruzando documentos de `produtos` com notas médias na coleção `avaliacoes`.

In [ ]:
pipeline_lookup = [
    {
        "$lookup": {
            "from": "avaliacoes",
            "localField": "_id",
            "foreignField": "produto_id",
            "as": "reviews"
        }
    },
    {
        "$project": {
            "sku": 1,
            "nome": 1,
            "categoria": 1,
            "preco": 1,
            "total_avaliacoes": {"$size": "$reviews"},
            "media_nota": {"$round": [{"$avg": "$reviews.nota"}, 1]}
        }
    },
    {"$sort": {"media_nota": -1}},
    {"$limit": 5}
]

pd.DataFrame(list(db.produtos.aggregate(pipeline_lookup)))